In [ ]:
%%capture
!pip install lightning adversarial-robustness-toolbox[pytorch_image]

In [72]:
from __future__ import absolute_import, division, print_function, unicode_literals, annotations
import os
import json
import sys
from pathlib import Path
from typing import Dict, List, Tuple, Optional, Union, TYPE_CHECKING
import logging
import time
from dataclasses import dataclass
from enum import Enum
from io import BytesIO
import logging


import h5py
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix
from tqdm.auto import tqdm
import pandas as pd

from einops import rearrange
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset, TensorDataset
import torchvision
from torchvision import transforms
import torchvision.models as models
import torchmetrics

from art.attacks.evasion import FastGradientMethod, ProjectedGradientDescent
from art.estimators.classification import PyTorchClassifier
from art.config import ART_NUMPY_DTYPE
from art.defences.preprocessor import *
from art.defences.preprocessor.preprocessor import Preprocessor

if TYPE_CHECKING:
    from art.utils import CLIP_VALUES_TYPE

import lightning as L
from lightning import LightningModule

In [62]:
class PCAMDataset(Dataset):
    """Optimized PCAM Dataset with lazy loading"""
    
    def __init__(self, input_file_path: str, label_file_path: str, 
                 transform=None, max_samples: Optional[int] = None):
        self.input_file_path = input_file_path
        self.label_file_path = label_file_path
        self.transform = transform
        self.max_samples = max_samples
        
        # Open files and keep references
        self.input_file = h5py.File(self.input_file_path, 'r')
        self.label_file = h5py.File(self.label_file_path, 'r')
        
        self.input_data = self.input_file["x"]
        self.label_data = self.label_file["y"]
        
        # Set length
        self.length = len(self.input_data)
        if max_samples is not None:
            self.length = min(max_samples, self.length)
            
    def __len__(self) -> int:
        return self.length
    
    def __getitem__(self, idx: int):
        if idx >= self.length:
            raise IndexError("Index out of range")
        
        # Load image and convert to PIL
        image = Image.fromarray(self.input_data[idx]).convert("RGB")
        label = int(self.label_data[idx, 0, 0, 0])
        
        if self.transform:
            image = self.transform(image)
            
        return image, label
    
    def __del__(self):
        # Close files when dataset is destroyed
        if hasattr(self, 'input_file'):
            self.input_file.close()
        if hasattr(self, 'label_file'):
            self.label_file.close()

In [63]:
class ResNet18Classifier(LightningModule):
    def __init__(self, num_classes: int = 2, label_smoothing: float = 0.0):
        super().__init__()
        self.model = models.resnet18(weights='DEFAULT')
        self.model.fc = nn.Linear(self.model.fc.in_features, num_classes)
        self.num_classes = num_classes
        
        self.criterion = nn.CrossEntropyLoss(label_smoothing=label_smoothing)

        self.train_acc = torchmetrics.Accuracy("binary", num_classes=num_classes)
        self.train_auroc = torchmetrics.AUROC("binary", num_classes=num_classes)
        self.val_acc = torchmetrics.Accuracy("binary", num_classes=num_classes)
        self.val_auroc = torchmetrics.AUROC("binary", num_classes=num_classes)
        self.test_acc = torchmetrics.Accuracy("binary", num_classes=num_classes)
        self.test_auroc = torchmetrics.AUROC("binary", num_classes=num_classes)

    def forward(self, x):
        return self.model(x)

    def _step(self, batch, stage: str):
        images, labels = batch

        outputs = self(images)
        loss = self.criterion(outputs, labels)
        preds = torch.argmax(outputs, dim=1)
        probs = F.softmax(outputs, dim=1)

        if stage == "train":
            self.train_acc(preds, labels)
            self.train_auroc(probs, labels)
        elif stage == "val":
            self.val_acc(preds, labels)
            self.val_auroc(probs, labels)
        elif stage == "test":
            self.test_acc(preds, labels)
            self.test_auroc(probs, labels)
        else:
            raise ValueError(f"Unknown stage: {stage}")
        
        self.log(f"{stage}_loss", loss, on_step=False, on_epoch=True, prog_bar=True)        
        return loss

    def training_step(self, batch, batch_idx):
        return self._step(batch, "train")
    def validation_step(self, batch, batch_idx):
        return self._step(batch, "val")
    def test_step(self, batch, batch_idx):
        return self._step(batch, "test")

    def on_train_epoch_end(self):
        self.log("train_acc", self.train_acc, on_step=False, on_epoch=True, prog_bar=True)
        self.log("train_auroc", self.train_auroc, on_step=False, on_epoch=True, prog_bar=True)
        self.train_acc.reset()
        self.train_auroc.reset()
    
    def on_validation_epoch_end(self):
        self.log("val_acc", self.val_acc, on_step=False, on_epoch=True, prog_bar=True)
        self.log("val_auroc", self.val_auroc, on_step=False, on_epoch=True, prog_bar=True)
        self.val_acc.reset()
        self.val_auroc.reset()

    def on_test_epoch_end(self):
        self.log("test_acc", self.test_acc, on_step=False, on_epoch=True, prog_bar=True)
        self.log("test_auroc", self.test_auroc, on_step=False, on_epoch=True, prog_bar=True)
        self.test_acc.reset()
        self.test_auroc.reset()

    def configure_optimizers(self):
        optimizer = optim.AdamW(self.parameters(), lr=1e-3, weight_decay=0.05)
        
        scheduler = optim.lr_scheduler.ReduceLROnPlateau(
            optimizer, mode='min', factor=0.1, patience=5
        )
        return {
            "optimizer": optimizer,
            "lr_scheduler": {
                "scheduler": scheduler,
                "monitor": "val_loss",
                "interval": "epoch",
                "frequency": 1,
            },
        }
        
    def freeze_backbone(self):
        for param in self.model.parameters():
            param.requires_grad = False
        for param in self.model.fc.parameters():  # Keep classifier trainable
            param.requires_grad = True

    def unfreeze_backbone(self):
        for param in self.model.parameters():
            param.requires_grad = True

In [156]:
# Configuration
DATA_ROOT = "/kaggle/input/metastatic-tissue-classification-patchcamelyon"
MODEL_PATH = "/kaggle/input/hermes_resnet_18/pytorch/lightning/1/reduced_dataset_10ep.ckpt"
MAX_SAMPLES = 5000  # Reduced for faster evaluation
BATCH_SIZE = 512
OUTPUT_DIR = Path("comprehensive_evaluation_results")
OUTPUT_DIR.mkdir(exist_ok=True)
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

In [157]:
# Data transforms
eval_transform = transforms.Compose([
    transforms.ToTensor(), # here they become [0, 1] due to torch backend with Pillow
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
])

In [158]:
# Load dataset
print("Loading test dataset...")
test_dataset = PCAMDataset(
    input_file_path=os.path.join(DATA_ROOT, "pcam/test_split.h5"),
    label_file_path=os.path.join(DATA_ROOT, "Labels/Labels/camelyonpatch_level_2_split_test_y.h5"),
    transform=eval_transform,
    max_samples=MAX_SAMPLES
)

Loading test dataset...


In [159]:
# Load model
print("Loading model...")
model = ResNet18Classifier.load_from_checkpoint(MODEL_PATH)

Loading model...


In [160]:
class GaussianSmoothing(Preprocessor):
    params = ["sigma", "verbose"]

    def __init__(
        self,
        clip_values: "CLIP_VALUES_TYPE" | None = None,
        sigma: float = 1.0,
        apply_fit: bool = True,
        apply_predict: bool = True,
        verbose: bool = False,
    ):
        """
        Some description of the class
        """
        super().__init__(is_fitted=True, apply_fit=apply_fit, apply_predict=apply_predict)
        self.sigma = sigma
        self.clip_values = clip_values
        self.verbose = verbose
        self._check_params()

    def __call__(self, x: np.ndarray, y: np.ndarray | None = None) -> tuple[np.ndarray, np.ndarray | None]:

        x_aug = np.random.normal(x, scale=self.sigma, size=x.shape).astype(ART_NUMPY_DTYPE)
        y_aug = y
        
        if self.clip_values is not None:
            x_aug = np.clip(x_aug, self.clip_values[0], self.clip_values[1])

        return x_aug, y_aug
        

    def _check_params(self) -> None:
        if not isinstance(self.sigma, float) or self.sigma <= 0:
            raise ValueError("Smoothing sigma must be a positive float.")

        if self.clip_values is not None:

            if len(self.clip_values) != 2:
                raise ValueError(
                    "`clip_values` should be a tuple of 2 floats or arrays containing the allowed data range."
                )
            if np.array(self.clip_values[0] >= self.clip_values[1]).any():
                raise ValueError("Invalid `clip_values`: min >= max.")

In [167]:
class UnNormalize(torch.nn.Module):
    def __init__(self, mean, std):
        super().__init__()
        self.mean = torch.tensor(mean).view(3,1,1)
        self.std  = torch.tensor(std ).view(3,1,1)
    
    def forward(self, tensor):
        # tensor is C×H×W, normalized
        return tensor * self.std + self.mean

class Rescale01(torch.nn.Module):
    """Rescale C×H×W tensor so its min→0 and max→1."""
    def forward(self, t):
        tmin, tmax = float(t.min()), float(t.max())
        if tmax <= tmin:
            return torch.zeros_like(t)
        return (t - tmin) / (tmax - tmin)

class TransformedDataset(Dataset):
    """Small wrapper to apply transform and do unnorm + 0-1 minmax scaling"""
    def __init__(self, base_dataset: Dataset, transform: Callable):
        self.base = base_dataset
        self.transform = transform

    def __len__(self):
        return len(self.base)

    def __getitem__(self, idx):
        x, y = self.base[idx]           # x: torch.Tensor C×H×W
        x_t = self.transform(x)
        return x_t, y

class MyFramework:
    """
    Modular framework for evaluating ART attacks and defenses.
    """
    def __init__(
        self,
        model: torch.nn.Module,
        test_dataset: torch.utils.data.Dataset,
        loss: torch.nn.Module,
        optimizer: torch.optim.Optimizer,
        input_shape: Tuple[int, int, int],
        num_classes: int,
        channels_first: bool,
        clip_values: Optional[Tuple[int, int]] = None,
        batch_size: int = 64
    ):
        # Core attributes
        self.model = model
        self.test_dataset = test_dataset
        self.loss = loss
        self.optimizer = optimizer
        self.input_shape = input_shape
        self.num_classes = num_classes
        self.channels_first = channels_first
        self.clip_values = clip_values
        self.batch_size = batch_size

        # Instantiate base classifier without defenses
        self.base_classifier = self._build_classifier(defences=None)

        # Registries for attacks and defenses
        self.attacks: List[Tuple[str, Callable[[art.attacks], Any]]] = []
        self.defenses: List[Tuple[str, Callable[[art.defences.preprocessor.preprocessor], Any]]] = []

    def _build_classifier(
        self,
        defences: Optional[List[Any]] = None
    ) -> PyTorchClassifier:
        """Return a new PyTorchClassifier with optional preprocessing defenses."""
        return PyTorchClassifier(
            model=self.model,
            loss=self.loss,
            optimizer=self.optimizer,
            input_shape=self.input_shape,
            nb_classes=self.num_classes,
            channels_first=self.channels_first,
            clip_values=self.clip_values,
            preprocessing_defences=defences
        )

    def _make_dataloader(self, dataset: torch.utils.data.Dataset) -> DataLoader:
        """Create a DataLoader for the given dataset."""
        return DataLoader(
            dataset,
            batch_size=self.batch_size,
            shuffle=False,
            pin_memory=True
        )

    def _generate_adversarial_dataset(
        self,
        classifier: PyTorchClassifier,
        attack_fn: Callable[[PyTorchClassifier], Any]
    ) -> torch.utils.data.Dataset:
        """Apply attack to test_dataset, return a TensorDataset of adversarial examples."""
        attack = attack_fn(classifier)
        loader = self._make_dataloader(self.test_dataset)

        adv_images, adv_labels = [], []
        for X, y in tqdm(loader, desc=f"Crafting adversarial ({attack.__class__.__name__})"):
            x_adv = attack.generate(x=X.numpy())
            adv_images.append(x_adv)
            adv_labels.append(y)

        X_adv = np.concatenate(adv_images, axis=0)
        y_adv = torch.cat(adv_labels)
        return TensorDataset(torch.from_numpy(X_adv), y_adv)

    def _evaluate_dataset(
        self,
        dataset: torch.utils.data.Dataset,
        classifier: PyTorchClassifier
    ) -> float:
        """Compute accuracy over the dataset using the given classifier."""
        loader = self._make_dataloader(dataset)
        accs = []
        for X, y in tqdm(loader, desc="Evaluating dataset"):
            preds = classifier.predict(X.numpy()).argmax(axis=1)
            accs.append(accuracy_score(y, preds))
        return float(np.mean(accs))

    def add_attack(
        self,
        name: str,
        attack_fn: Callable[[PyTorchClassifier], Any]
    ):
        """Register an attack factory."""
        self.attacks.append((name, attack_fn))

    def add_defense(
        self,
        name: str,
        defense_fn: Callable[[], Any]
    ):
        """Register a defense factory."""
        self.defenses.append((name, defense_fn))

    def run_experiments(self) -> pd.DataFrame:
        """Run all combinations of clean, attacks, and defenses, return results DataFrame."""
        results = []

        # Clean data
        clean_acc = self._evaluate_dataset(self.test_dataset, self.base_classifier)
        results.append({"attack": "none", "defense": "none", "accuracy": clean_acc})

        # Loop over attacks
        for atk_name, atk_fn in self.attacks:
            # Generate adversarial set
            adv_ds = self._generate_adversarial_dataset(self.base_classifier, atk_fn)

            # Evaluate adv without defense
            atk_acc = self._evaluate_dataset(adv_ds, self.base_classifier)
            results.append({"attack": atk_name, "defense": "none", "accuracy": atk_acc})

            # Evaluate adv with each defense
            # some specific defenses require attention in their expected input
            for def_name, def_fn in self.defenses:
                clf_def = self._build_classifier(defences=[def_fn()])
                # if doing JPEG Compression, input X must be [0,1] range or [0,255]
                if 'jpeg' in def_name.lower():
                    unnorm = UnNormalize(IMAGENET_MEAN, IMAGENET_STD) # unnormalize class

                    # compose the transforms needed
                    jpeg_preproc = transforms.Compose([
                        unnorm, # undo mean/std
                        Rescale01(), # back to [0,1] since unnorm on adversarial leads to slightly lower than 0 and higher than 1 values
                    ])

                    # create a new dataset with that transform
                    jpeg_ds = TransformedDataset(adv_ds, jpeg_preproc) 
                    def_acc = self._evaluate_dataset(jpeg_ds, clf_def)
                    
                # for other cases we just evaluate the defense on the adversarial examples
                else:  
                    def_acc = self._evaluate_dataset(adv_ds, clf_def)
                results.append({"attack": atk_name, "defense": def_name, "accuracy": def_acc})

        return pd.DataFrame(results)

    def plot_example(self, example: torch.Tensor, original: Optional[torch.Tensor] = None):
        # Rearrange and convert the example tensor to a NumPy array
        img = rearrange(example, 'C H W -> H W C').numpy().squeeze()
    
        if original is not None:
            # Rearrange and convert the original tensor to a NumPy array
            orig_img = rearrange(original, 'C H W -> H W C').numpy().squeeze()
    
            # Create a figure with two subplots side by side
            fig, axes = plt.subplots(1, 2, figsize=(10, 5))
    
            # Display the original image
            axes[0].imshow(orig_img)
            axes[0].set_title('Original')
            axes[0].axis('off')
    
            # Display the processed image
            axes[1].imshow(img)
            axes[1].set_title('Processed')
            axes[1].axis('off')
    
            plt.tight_layout()
            plt.show()
        else:
            # Display only the processed image
            plt.imshow(img)
            plt.axis('off')
            plt.title('Processed')
            plt.show()


Instantiate the framework

In [106]:
frame = MyFramework(
    model=model.model,
    test_dataset=test_dataset,
    loss=nn.CrossEntropyLoss(),
    optimizer=optim.AdamW(model.model.parameters(), lr=1e-3),
    input_shape=(3, 96, 96),
    num_classes=2,
    channels_first=True,
    clip_values=None,
    batch_size=BATCH_SIZE
)

Run evaluation with the base classifier on the clean test dataset. Pass the classifier to use

In [ ]:
frame._evaluate_dataset(dataset=test_dataset, classifier=frame.base_classifier)

Craft some adverarial examples (still need to modularize the kind of attacks and parameters). Pass the dataset to use and the classifier to use

In [ ]:
adversarial_dataset = frame._generate_adversarial_dataset(
    classifier=frame.base_classifier, 
    attack_fn=FastGradientMethod
)

Visualize the adversarial example and the original one or just one of them if original is not specified

In [ ]:
idx = 2
frame.plot_example(adversarial_dataset[idx][0], original=test_dataset[idx][0])

Run evaluation of the base classifier on the adversarial dataset given by applying the attack over the clean test dataset

In [ ]:
frame._evaluate_dataset(dataset=adversarial_dataset, classifier=frame.base_classifier)

Now, create another classifier which incorporates one or more defense strategies

In [ ]:
defense_classifier = frame._build_classifier(
    defences=[GaussianSmoothing(sigma=1.0)]
)

Run evaluation on the dataset with adversarial examples using the defense classifier rather than the base classifier. The defense classifier has inside some defences

In [ ]:
frame._evaluate_dataset(dataset=adversarial_dataset, classifier=defense_classifier)

Now, try the full pipeline. First instantiate a clean framework and then add attacks and defenses

In [173]:
clean_frame = MyFramework(
    model=model.model,
    test_dataset=test_dataset,
    loss=nn.CrossEntropyLoss(),
    optimizer=optim.AdamW(model.model.parameters(), lr=1e-3),
    input_shape=(3, 96, 96),
    num_classes=2,
    channels_first=True,
    clip_values=None,
    batch_size=BATCH_SIZE
)

Add attacks

In [174]:
clean_frame.add_attack("FGSM (eps=0.2)", lambda clf: FastGradientMethod(estimator=clf, eps=0.2))
#clean_frame.add_attack("FGSM (eps=1.0)", lambda clf: FastGradientMethod(estimator=clf, eps=1.0))
#clean_frame.add_attack("PGD (eps=0.3)", lambda clf: ProjectedGradientDescent(estimator=clf, eps=0.3, max_iter=40))
#clean_frame.add_attack("PGD (eps=0.7)", lambda clf: ProjectedGradientDescent(estimator=clf, eps=0.3, max_iter=40))

Add defenses

In [175]:
#clean_frame.add_defense("GaussianSmooth (sigma=0.5)", lambda: GaussianSmoothing(sigma=0.5))
#clean_frame.add_defense("GaussianSmooth (sigma=2.0)", lambda: GaussianSmoothing(sigma=2.0))
clean_frame.add_defense("JpegCompression", lambda: JpegCompression(clip_values=(0,1), quality=95, channels_first=True))

The run experiments works in this way:

- creates an empty object to store the metrics
- evaluates the clean test dataset with the base classifier
- loops over the defined attacks (you need to add them, check above)
- for each attack it generates a torch dataset applying the attack on each sample and then evaluates the adversarial crafted test set with base classifier
- then it loops over the defenses and for each it builds the classifier using only that defense and then it evaluates the adversarial crafted test set with the classifier having that defense as countermeasure.


In [176]:
results_df = clean_frame.run_experiments()

Evaluating dataset:   0%|          | 0/10 [00:00<?, ?it/s]

Crafting adversarial (FastGradientMethod):   0%|          | 0/10 [00:00<?, ?it/s]

Evaluating dataset:   0%|          | 0/10 [00:00<?, ?it/s]

Evaluating dataset:   0%|          | 0/10 [00:00<?, ?it/s]

In [178]:
results_df

,attack,defense,accuracy
0,none,none,0.851566
1,FGSM (eps=0.2),none,0.233474
2,FGSM (eps=0.2),JpegCompression,0.497409
